# Name That LLM | Part Two residual readouts

Hypothesis: content-related variation in late-layer features may hide author style. Compare an ordinary regularized linear readout with a content-residual readout, then test a fixed 10% blend on three prompt-grouped folds. The new helper is used only if the predeclared proxy gate passes. All scaling, PCA, text vocabulary, SVD, content regression and classifier fitting happen inside each fold.

Both versions extract the same test and official-training features with the same three transformer models. This version adds CPU readout fitting, not another backbone. On gate rejection its selected submission is the Part Two baseline. Fixed backbone features come from full-fit adapters: this is **not full-pipeline OOF**, and proxy gains do not guarantee leaderboard gains.

## Credits and provenance

The baseline adapts **Nishant Dahal's [Two-stage LoRA, part two](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two)** and its [NTL two-stage LoRA part two assets](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend). The source reports public LB 0.620; this adaptation has not reproduced that score. Source code SHA256 is recorded below. Asset metadata and linear heads are pinned by SHA256; adapter file sizes and tensor shapes, base architecture, and all weight shards are checked before inference.

Recipe credits: [Vladislav Lavrentev](https://www.kaggle.com/lightningv08) and [Lucas Boesen](https://www.kaggle.com/lucasboesen) for the upstream LoRA24 recipe, adaptive centering and balancing; [yangkuangou](https://www.kaggle.com/yangkuangou) for style-aware n-gram views. Qwen models: Qwen team, Apache-2.0. DeBERTa: Microsoft, MIT. The author publishes the combined inference assets/pool under **CC BY-SA 4.0**; retain their dataset attribution and upstream licenses.

The research readout in iteration 12 is our own implementation. It is inspired by the general prompt-residual idea, not a reproduction of the private Night24 banks in the supplied Spectral/Synthetic notebooks. Those private banks, synthetic passages and ATLAS are not required.


## Attach these inputs before running

Both iterations use the **same four inputs**. The four versioned inputs are embedded in this notebook's Kaggle dataSources metadata and verified through the Kaggle API on 2026-10-24. After importing the updated file, confirm all four entries appear in the Input panel before running. Some import flows may keep the existing editor inputs; use the links below if Kaggle does not apply the embedded metadata.

| Kaggle section | Required input | Revision / expected files |
|---|---|---|
| Competition | [Name That LLM](https://www.kaggle.com/competitions/name-that-llm) | `train.csv`, `test.csv` |
| Dataset | [nishantdahal/ntl-handwriting-blend](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend) | **Dataset version 1**; `pool_v4.parquet`, `lora_qwen3_14b`, `lora_qwen25_14b`, `deberta_v3_large` |
| Models | [Qwen3-14B-Base](https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b-base/1) | Transformers, **14b-base, version 1** |
| Models | [Qwen2.5-14B](https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/14b/1) | Transformers, **14b, version 1** |

Use GPU **T4 x2** and Internet **off** for inference. Qwen2.5 **7B** and the older `ntl-two-stage-lora` dataset cannot replace these inputs. No notebook-output dependencies or private sources are needed. Fresh notebooks with only these inputs avoid ambiguity and repeated mount scans. The input manifest below records requests; it is not a downloader.

For a CPU-only input check, choose accelerator None and set `config.run_inference = False` before the preflight cells. This checks files, metadata, heads, tokenizers and data without loading transformer weights into a model; it cannot validate CUDA inference. The verified asset metadata is pinned: a checksum mismatch means the attached version must be reviewed, not bypassed.


In [ ]:
'''1. Offline environment
The source used Kaggle image sha256:9a6bcfda6d03b2cd9aa347ee8c1bae62043328ed0f4afe9bda014e4f8bbf9435.
Use its package environment where available; this notebook does not install or upgrade libraries.
No three-row shortcut: every test row receives real inference. A visible run is not a hidden runtime benchmark.'''

In [ ]:
import os
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[key] = '4'
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')

In [ ]:
import gc
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import struct
import tempfile
import time
from contextlib import contextmanager
import numpy as np
import pandas as pd
import torch
from torch import nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, RidgeClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

In [ ]:
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
LORAS = {'lora_qwen3_14b': dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type='qwen3', hidden_size=5120, num_hidden_layers=40)), 'lora_qwen25_14b': dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type='qwen2', hidden_size=5120, num_hidden_layers=48))}
SMALL_WORDS = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('small', 1, 3, 50000, 0.5)]
INPUT_URLS = {'competition': 'https://www.kaggle.com/competitions/name-that-llm', 'assets': 'https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend', 'lora_qwen3_14b': 'https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b-base/1', 'lora_qwen25_14b': 'https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/14b/1'}
PINNED_SHA256 = {'lora_qwen3_14b': {'meta.json': '293119872ff3f170ec85e8be02c80916e39048213de31cb45f8b84a2cf74d775', 'heads.safetensors': 'e1d9c1ac833a1cc780f1423341bb5299826271f5aeed2c40d3ec193633d45faf'}, 'lora_qwen25_14b': {'meta.json': '1d04028b7fe22beda04f7125d09dd5bc9bee9c8524f721c0aa7bc61379fad6fc', 'heads.safetensors': '73762ab515ed984efc9189063ee4efddd1a0b0b2620aab6420b3e56e1a1c752e'}, 'deberta_v3_large': {'meta.json': 'deb3c891d6172018ec06b7b3de33f9a13ac6c44d6bf8a9725b53ce8c7c93a546', 'head.safetensors': '43db1fbe725abfc2299e6618f2f6b3e7cb7a63e3981cb4325dd6961e6e1afb06'}}
EXPECTED_ADAPTER_BYTES = {'lora_qwen3_14b': 173456688, 'lora_qwen25_14b': 183558248}
BASE_PATHS = {'lora_qwen3_14b': 'qwen-lm/qwen-3/transformers/14b-base/1', 'lora_qwen25_14b': 'qwen-lm/qwen2.5/transformers/14b/1'}
SOURCE_URL = 'https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two'
SOURCE_SHA256 = '6a33ee530a46129746e8b06588829e50fc13a67639f019d378ffd44ff963dee1'

In [ ]:
'''2. Configuration
Leave training_cache_root as None on a first run.
An optional cache may contain only verified official-training features exported by iteration 11 or 12.
Test predictions are never loaded from another notebook; hidden inference always processes the current test.'''

In [ ]:
class ExperimentConfig:
    def __init__(self, research=False):
        self.research = research
        self.run_inference = True
        self.input_root = Path('/kaggle/input')
        self.output_root = Path('/kaggle/working')
        self.scratch_root = Path('/kaggle/temp/ntl_part_two')
        self.training_cache_root = None
        self.labels = list(LABELS)
        self.loras = LORAS
        self.readout_weight = 0.10
        self.seed = 42

In [ ]:
config = ExperimentConfig(research=True)
'''Set run_inference to False here for a CPU-only preflight.'''
config.run_inference = True

In [ ]:
'''3. Exact inputs and structural checks
Inspect the printed EOS and resolved model paths. Config-only folders and missing shards are rejected.
Metadata and head checksums pin the reviewed assets; base model weights are structurally checked, not fully hashed.'''

In [ ]:
class FileChecks:
    @staticmethod
    def sha256(path):
        with Path(path).open('rb') as stream:
            return hashlib.file_digest(stream, 'sha256').hexdigest()

    @staticmethod
    def tensor_shapes(path):
        with Path(path).open('rb') as stream:
            prefix = stream.read(8)
            if len(prefix) != 8:
                raise ValueError(f'Invalid safetensors header: {path}')
            size = struct.unpack('<Q', prefix)[0]
            if not 2 <= size <= 16 * 1024 * 1024:
                raise ValueError(f'Unexpected safetensors header size: {path}')
            header = json.loads(stream.read(size))
        return {key: value['shape'] for key, value in header.items() if key != '__metadata__'}

    @staticmethod
    def weight_files(directory):
        directory = Path(directory)
        for name in ('model.safetensors', 'pytorch_model.bin'):
            path = directory / name
            if path.is_file() and path.stat().st_size > 0:
                return [path]
        for name in ('model.safetensors.index.json', 'pytorch_model.bin.index.json'):
            index = directory / name
            if index.is_file():
                mapping = json.loads(index.read_text())['weight_map']
                paths = [directory / part for part in sorted(set(mapping.values()))]
                if paths and all(path.is_file() and path.stat().st_size > 0 for path in paths):
                    return paths
        raise FileNotFoundError(f'No complete model weights in {directory}; config-only folders are not checkpoints')

In [ ]:
class InputAssets:
    def __init__(self, config):
        self.config = config
        self.assets = None
        self.bases = {}
        self.manifest = {}

    def resolve_base(self, key, spec):
        relative = BASE_PATHS[key]
        roots = [self.config.input_root / 'models' / relative, self.config.input_root / relative,
                 self.config.input_root / Path(*Path(relative).parts[1:])]
        candidates = [root for root in roots if (root / 'config.json').is_file()]
        if not candidates:
            raise FileNotFoundError(f'Attach the exact model: {INPUT_URLS[key]}. Expected {relative}')
        if len(candidates) != 1:
            raise ValueError(f'Multiple mounted copies of {key}; attach only the requested model revision')
        path = candidates[0]
        raw = json.loads((path / 'config.json').read_text())
        if any(raw.get(field) != value for field, value in spec.items()):
            raise ValueError(f'Wrong architecture for {key}: {path}')
        files = FileChecks.weight_files(path)
        tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        meta = json.loads((self.assets / key / 'meta.json').read_text())
        eos = meta['cfg']['eos_id']
        if not 0 <= eos < len(tokenizer):
            raise ValueError(f'Training EOS {eos} is outside the tokenizer vocabulary for {key}')
        self.manifest[key] = dict(base=str(path), config_sha256=FileChecks.sha256(path / 'config.json'),
            training_eos_id=eos, tokenizer_eos_id=tokenizer.eos_token_id,
            weights=[dict(name=f.name, bytes=f.stat().st_size) for f in files])
        return path

    def validate_lora(self, key, spec):
        directory = self.assets / key
        for relative in ['meta.json', 'heads.safetensors', 'adapter/adapter_config.json', 'adapter/adapter_model.safetensors']:
            path = directory / relative
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f'Missing {path}. Attach {INPUT_URLS["assets"]}')
        for filename, expected in PINNED_SHA256[key].items():
            if FileChecks.sha256(directory / filename) != expected:
                raise ValueError(f'{key}/{filename} differs from the verified Part Two asset; review the dataset version')
        meta = json.loads((directory / 'meta.json').read_text())
        if meta['labels'] != self.config.labels:
            raise ValueError(f'Checkpoint label order mismatch: {key}')
        settings = meta['cfg']
        if settings.get('eos_id') != 151643 or not 0 < settings['keep'] <= spec['base']['num_hidden_layers']:
            raise ValueError(f'Unexpected training EOS or retained layer count: {key}')
        if len(settings['layers']) != 8 or min(settings['layers']) < 1 or max(settings['layers']) > settings['keep']:
            raise ValueError(f'Invalid pooling layers in {key}')
        adapter = json.loads((directory / 'adapter/adapter_config.json').read_text())
        if adapter['r'] != settings['r'] or adapter['lora_alpha'] != settings['alpha']:
            raise ValueError(f'Adapter and training metadata differ: {key}')
        hidden = spec['base']['hidden_size']
        shapes = FileChecks.tensor_shapes(directory / 'heads.safetensors')
        for prefix in ['plain', 'centered']:
            if shapes.get(prefix + '.weight') != [len(self.config.labels), hidden * 8] or shapes.get(prefix + '.bias') != [len(self.config.labels)]:
                raise ValueError(f'Invalid {prefix} head dimensions for {key}')
        for layer in range(8):
            for suffix in ['weight', 'bias']:
                if shapes.get(f'norms.{layer}.{suffix}') != [hidden]:
                    raise ValueError(f'Invalid pooling normalization for {key}')
        for tensor, shape in FileChecks.tensor_shapes(directory / 'adapter/adapter_model.safetensors').items():
            if not shape or settings['r'] not in shape:
                raise ValueError(f'Unexpected LoRA tensor shape: {key}/{tensor}')
        if (directory / 'adapter/adapter_model.safetensors').stat().st_size != EXPECTED_ADAPTER_BYTES[key]:
            raise ValueError(f'Unexpected adapter file size for {key}; check the dataset version')

    def preflight(self):
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        roots = [self.config.input_root / 'datasets/nishantdahal/ntl-handwriting-blend',
                 self.config.input_root / 'ntl-handwriting-blend']
        candidates = [root for root in roots if (root / 'pool_v4.parquet').is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(f'Attach one copy of {INPUT_URLS["assets"]}; old ntl-two-stage-lora assets are incompatible')
        self.assets = candidates[0]
        for key, spec in self.config.loras.items():
            self.validate_lora(key, spec)
            self.bases[key] = self.resolve_base(key, spec['base'])
        deb = self.assets / 'deberta_v3_large'
        for name, expected in PINNED_SHA256['deberta_v3_large'].items():
            if not (deb / name).is_file() or FileChecks.sha256(deb / name) != expected:
                raise ValueError(f'Missing or incompatible DeBERTa asset: {name}')
        meta = json.loads((deb / 'meta.json').read_text())
        if meta['labels'] != self.config.labels or meta['max_len'] != 384:
            raise ValueError('DeBERTa label order or token limit differs from the source')
        if FileChecks.tensor_shapes(deb / 'head.safetensors') != {'bias': [18], 'weight': [18, 1024]}:
            raise ValueError('Invalid DeBERTa head')
        FileChecks.weight_files(deb / 'encoder')
        AutoTokenizer.from_pretrained(str(deb / 'encoder'), local_files_only=True)
        self.manifest['adapter_sha256'] = {key: FileChecks.sha256(self.assets / key / 'adapter/adapter_model.safetensors')
                                           for key in self.config.loras}
        self.manifest['asset_sha256'] = PINNED_SHA256
        self.manifest['source_code_sha256'] = SOURCE_SHA256
        (self.config.output_root / 'input_manifest.json').write_text(json.dumps(self.manifest, indent=2))
        print('INPUT CHECKS PASSED: exact Part Two assets, complete base weights, 18 labels, training EOS 151643', flush=True)
        return self

    def load_data(self):
        roots = [self.config.input_root / 'competitions/name-that-llm', self.config.input_root / 'name-that-llm']
        candidates = [root for root in roots if (root / 'train.csv').is_file() and (root / 'test.csv').is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(f'Attach the competition data: {INPUT_URLS["competition"]}')
        directory = candidates[0]
        train = pd.read_csv(directory / 'train.csv', dtype={'id': str})
        test = pd.read_csv(directory / 'test.csv', dtype={'id': str})
        if not {'id', 'text', 'model', 'prompt_id'}.issubset(train) or not {'id', 'text'}.issubset(test):
            raise ValueError('Unexpected competition CSV schema')
        for frame in [train, test]:
            if frame.empty or frame['id'].isna().any() or frame['id'].duplicated().any():
                raise ValueError('Expected nonempty CSVs with unique non-null IDs')
        if set(train['model']) != set(self.config.labels) or train[['text', 'prompt_id']].isna().any().any():
            raise ValueError('Missing training text, prompt IDs or author labels')
        test['text'] = test['text'].fillna('').astype(str)
        pool = pd.read_parquet(self.assets / 'pool_v4.parquet')
        if not {'text', 'model'}.issubset(pool) or len(pool) != 75783:
            raise ValueError('Expected the verified 75,783-row pool_v4; check the dataset version')
        if pool[['text', 'model']].isna().any().any() or not pool['model'].isin(self.config.labels).all():
            raise ValueError('Invalid public-pool text or author labels')
        self.data_hashes = {name: FileChecks.sha256(directory / f'{name}.csv') for name in ['train', 'test']}
        return train, test, pool

    def feature_signature(self):
        record = dict(train_sha256=self.data_hashes['train'], labels=self.config.labels, assets=PINNED_SHA256,
                      adapters=self.manifest['adapter_sha256'],
                      bases={key: {field: value for field, value in record.items() if field != 'base'}
                             for key, record in self.manifest.items() if key in self.config.loras})
        return hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()

In [ ]:
'''4. Timing, local staging and source probability calibration'''

In [ ]:
class StageTimer:
    def __init__(self, root):
        self.path = root / 'stage_timings.json'
        self.records = []

    @contextmanager
    def measure(self, name):
        record = {'stage': name, 'status': 'running'}
        self.records.append(record)
        self.path.write_text(json.dumps(self.records, indent=2))
        print(f'START {name}', flush=True)
        started = time.monotonic()
        try:
            yield
        except BaseException:
            record['status'] = 'failed'
            raise
        else:
            record['status'] = 'complete'
        finally:
            record['seconds'] = time.monotonic() - started
            self.path.write_text(json.dumps(self.records, indent=2))
            print(f"END {name}: {record['status']}, {record['seconds'] / 60:.1f} min", flush=True)

In [ ]:
class LocalCheckpointStage:
    @staticmethod
    def copy(source, target):
        source, target = Path(source), Path(target)
        if target.exists():
            raise FileExistsError(f'Use a fresh session; scratch checkpoint already exists: {target}')
        files = [path for path in source.rglob('*') if path.is_file()]
        target.parent.mkdir(parents=True, exist_ok=True)
        total = sum(path.stat().st_size for path in files)
        if shutil.disk_usage(target.parent).free < total + 2**30:
            raise OSError('Insufficient scratch space to stage checkpoint before GPU loading')
        for path in files:
            output = target / path.relative_to(source)
            output.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, output)
            if output.stat().st_size != path.stat().st_size:
                raise OSError(f'Incomplete staged checkpoint file: {output}')
        print(f'Staged {total / 2**20:.0f} MiB: {source.name}', flush=True)
        return target

In [ ]:
class ProbabilityMath:
    @staticmethod
    def softmax(values):
        exponential = np.exp(values - values.max(1, keepdims=True))
        return exponential / exponential.sum(1, keepdims=True)

    @staticmethod
    def check(probabilities, rows):
        if probabilities.shape != (rows, len(LABELS)) or not np.isfinite(probabilities).all() or (probabilities < 0).any():
            raise ValueError('Invalid probability matrix')
        if not np.allclose(probabilities.sum(1), 1, atol=1e-5):
            raise ValueError('Probability rows must sum to one')

    @staticmethod
    def balance(probabilities, iters=300):
        probabilities = probabilities.copy()
        if len(probabilities) < 50:
            return probabilities
        scales = np.ones(len(LABELS))
        target = np.full(len(LABELS), len(probabilities) / len(LABELS))
        for _ in range(iters):
            result = probabilities * scales
            result /= result.sum(1, keepdims=True)
            scales *= target / result.sum(0)
        result = probabilities * scales
        return result / result.sum(1, keepdims=True)

In [ ]:
class NotebookContractChecks:
    @staticmethod
    def run():
        rng = np.random.RandomState(42)
        probabilities = ProbabilityMath.softmax(rng.normal(size=(90, len(LABELS))))
        balanced = ProbabilityMath.balance(probabilities)
        ProbabilityMath.check(balanced, 90)
        np.testing.assert_allclose(balanced.sum(0), np.full(len(LABELS), 90 / len(LABELS)), atol=1e-5)
        np.testing.assert_array_equal(ProbabilityMath.balance(probabilities[:3]), probabilities[:3])
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory)
            (path / 'config.json').write_text('{}')
            try:
                FileChecks.weight_files(path)
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('A config-only folder was accepted')
            (path / 'model.safetensors.index.json').write_text(json.dumps({'weight_map': {'a': 'one', 'b': 'two'}}))
            (path / 'one').write_bytes(b'fixture')
            try:
                FileChecks.weight_files(path)
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('An incomplete sharded checkpoint was accepted')
            (path / 'two').write_bytes(b'fixture')
            assert len(FileChecks.weight_files(path)) == 2

NotebookContractChecks.run()


In [ ]:
'''5. Run input checks before any expensive model inference'''

In [ ]:
assets = InputAssets(config).preflight()
train, test, pool = assets.load_data()
print(f"Train: {len(train)}, test: {len(test)}, public pool: {len(pool)}")

In [ ]:
'''6. Source style classifier and average-linkage topic groups'''

In [ ]:
class StyleClassifier:
    @staticmethod
    def shape_text(text):
        return re.sub(r'[^\W\d_]+', lambda match: 'X' if match[0].isupper() else 'x', text)

    @staticmethod
    def function_text(text):
        return ' '.join(word.lower() if word.lower() in SMALL_WORDS or not word.isalnum() else 'CONTENT'
                        for word in re.findall(r'\w+|[^\w\s]', text))

    def fit_predict(self, train, test, pool):
        texts = train['text'].astype(str).tolist() + pool['text'].tolist()
        mapping = {label: index for index, label in enumerate(LABELS)}
        y = np.r_[train['model'].map(mapping).to_numpy(), pool['model'].map(mapping).to_numpy()]
        training, query = [], []
        for view, low, high, maximum, weight in VIEWS:
            prepare = self.shape_text if view == 'shape' else self.function_text if view == 'small' else str
            analyzer = 'char' if view == 'shape' else 'word' if view == 'small' else view
            options = dict(analyzer=analyzer, ngram_range=(low, high), max_features=maximum,
                           lowercase=False, sublinear_tf=True, min_df=2, dtype=np.float32)
            if analyzer == 'word':
                options['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
            vectorizer = TfidfVectorizer(**options)
            training.append(vectorizer.fit_transform([prepare(text) for text in texts]) * weight)
            query.append(vectorizer.transform([prepare(text) for text in test['text']]) * weight)
        classifier = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
        classifier.fit(hstack(training, format='csr'), y)
        if not np.array_equal(classifier.classes_, np.arange(len(LABELS))):
            raise ValueError('Unexpected SVC class order')
        return ProbabilityMath.softmax(classifier.decision_function(hstack(query, format='csr')))

In [ ]:
class TopicGrouping:
    @staticmethod
    def detect(texts):
        if len(texts) < 2:
            groups = np.ones(len(texts), dtype=int)
        else:
            features = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(list(texts))
            distance = np.clip(1 - (features @ features.T).toarray(), 0, None)
            np.fill_diagonal(distance, 0)
            groups = fcluster(linkage(squareform(distance, checks=False), method='average'), 0.85, criterion='distance')
        sizes = np.bincount(groups)[groups]
        return groups, (sizes >= 2) & (sizes <= 18)

In [ ]:
'''7. LoRA pooling and source heads
Training EOS is mandatory; no fallback to a tokenizer default.
Test and train batches are pooled separately on the same loaded model.
The test pooling follows the author arithmetic; the last pooled training layer is exported for CPU research.'''

In [ ]:
class LoRAFeatureExtractor:
    def __init__(self):
        self.token_audit = []

    def load_trunk(self, base, keep, dtype):
        cfg = AutoConfig.from_pretrained(str(base), local_files_only=True)
        cfg.num_hidden_layers, cfg.use_cache = keep, False
        if getattr(cfg, 'layer_types', None) is not None:
            cfg.layer_types = cfg.layer_types[:keep]
        hidden, intermediate, vocab = cfg.hidden_size, cfg.intermediate_size, cfg.vocab_size
        head_dim = getattr(cfg, 'head_dim', hidden // cfg.num_attention_heads)
        q, kv = cfg.num_attention_heads * head_dim, cfg.num_key_value_heads * head_dim
        gib = 2 * (vocab * hidden + keep * (2 * hidden * q + 2 * hidden * kv + 3 * hidden * intermediate)) / 2**30
        options = dict(config=cfg, dtype=dtype, local_files_only=True)
        if gib > 13:
            options.update(device_map='auto', max_memory={0: '13GiB', 1: '13GiB'})
            model = AutoModel.from_pretrained(str(base), **options).eval()
            if any(str(device) in {'cpu', 'disk'} for device in getattr(model, 'hf_device_map', {}).values()):
                raise RuntimeError('CPU/disk model offload detected; stop rather than run an unexpectedly slow model')
        else:
            model = AutoModel.from_pretrained(str(base), **options).to('cuda:0').eval()
        model.norm = nn.Identity()
        return model

    @torch.no_grad()
    def merge(self, model, directory):
        config = json.loads((directory / 'adapter/adapter_config.json').read_text())
        state = load_file(str(directory / 'adapter/adapter_model.safetensors'))
        modules = dict(model.named_modules())
        for name, left in state.items():
            if '.lora_A.' in name:
                prefix = name.split('.lora_A.')[0]
                layer = modules[prefix[prefix.index('layers.'):]]
                right = state[name.replace('.lora_A.', '.lora_B.')]
                delta = (right.float().to(layer.weight.device) @ left.float().to(layer.weight.device)) * (config['lora_alpha'] / config['r'])
                layer.weight.add_(delta.to(layer.weight.dtype))

    @torch.no_grad()
    def pool_batch(self, indices, ids, model, tokenizer, layers, norms, dtype, output):
        batch, hidden_states = None, None
        try:
            batch = tokenizer.pad({'input_ids': [ids[index] for index in indices]}, return_tensors='pt').to(model.embed_tokens.weight.device)
            hidden_states = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hidden_states[layer]).all() for layer in layers):
                hidden_states = None
                model.to(torch.bfloat16)
                hidden_states = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
                if not all(torch.isfinite(hidden_states[layer]).all() for layer in layers):
                    raise FloatingPointError('Non-finite hidden states after the bfloat16 retry')
            mask = batch['attention_mask'].unsqueeze(-1).float().to('cuda:0')
            mask[:, 0] = 0
            denominator = mask.sum(1).clamp(min=1)
            output[indices] = torch.cat([normalizer((hidden_states[layer].to('cuda:0').float() * mask).sum(1) / denominator)
                                        for layer, normalizer in zip(layers, norms)], -1).cpu().numpy()
        finally:
            del hidden_states, batch

    def pool(self, texts, model, tokenizer, settings, norms, dtype, batch_size, name):
        ids = tokenizer(list(texts), add_special_tokens=False, truncation=True, max_length=settings['max_len'])['input_ids']
        ids = [tokens + [settings['eos_id']] for tokens in ids]
        order = np.argsort([len(tokens) for tokens in ids], kind='stable')
        output = np.zeros((len(ids), model.config.hidden_size * len(settings['layers'])), np.float32)
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            try:
                self.pool_batch(indices, ids, model, tokenizer, settings['layers'], norms, dtype, output)
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                for index in indices:
                    self.pool_batch([index], ids, model, tokenizer, settings['layers'], norms, dtype, output)
            if start % (25 * batch_size) == 0 or start + batch_size >= len(order):
                print(f'{name}: {min(start + batch_size, len(order))}/{len(order)} passages', flush=True)
        if not np.isfinite(output).all():
            raise FloatingPointError(f'Invalid pooled features: {name}')
        return output

    def extract(self, directory, base, test_texts, train_texts, batch_size):
        settings = json.loads((directory / 'meta.json').read_text())['cfg']
        tokenizer = AutoTokenizer.from_pretrained(str(base), local_files_only=True)
        tokenizer.padding_side = 'right'
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.token_audit.append(dict(checkpoint=directory.name, training_eos_id=settings['eos_id'], tokenizer_eos_id=tokenizer.eos_token_id))
        print(f"EOS {directory.name}: training={settings['eos_id']}, tokenizer={tokenizer.eos_token_id}", flush=True)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        model = self.load_trunk(base, settings['keep'], dtype)
        try:
            self.merge(model, directory)
            heads = load_file(str(directory / 'heads.safetensors'))
            hidden = model.config.hidden_size
            norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in settings['layers']]).to('cuda:0').float()
            norms.load_state_dict({key[len('norms.'):]: value for key, value in heads.items() if key.startswith('norms.')})
            query = self.pool(test_texts, model, tokenizer, settings, norms, dtype, batch_size, directory.name + ' test')
            training = (self.pool(train_texts, model, tokenizer, settings, norms, dtype, batch_size, directory.name + ' train')[:, -hidden:].copy()
                        if train_texts is not None else None)
            readouts = {key: value.float().numpy() for key, value in heads.items() if not key.startswith('norms.')}
            return query, training, readouts
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
'''8. Fully fine-tuned DeBERTa
Stage the encoder locally before loading to avoid many reads from the dataset mount.'''

In [ ]:
class DebertaClassifier:
    @torch.no_grad()
    def predict(self, directory, texts):
        metadata = json.loads((directory / 'meta.json').read_text())
        tokenizer = AutoTokenizer.from_pretrained(str(directory / 'encoder'), local_files_only=True)
        model = AutoModel.from_pretrained(str(directory / 'encoder'), dtype=torch.float32, local_files_only=True).to('cuda:0').eval()
        try:
            head = load_file(str(directory / 'head.safetensors'))
            ids = tokenizer(list(texts), truncation=True, max_length=metadata['max_len'])['input_ids']
            order = np.argsort([len(tokens) for tokens in ids], kind='stable')
            output = np.zeros((len(ids), model.config.hidden_size), np.float32)
            for start in range(0, len(order), 32):
                rows = order[start:start + 32]
                batch = tokenizer.pad({'input_ids': [ids[index] for index in rows]}, return_tensors='pt').to('cuda:0')
                mask = batch['attention_mask'].unsqueeze(-1).float()
                states = model(**batch).last_hidden_state
                output[rows] = ((states * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()
                del batch, mask, states
                print(f'DeBERTa: {min(start + 32, len(order))}/{len(order)} passages', flush=True)
            return ProbabilityMath.softmax(output @ head['weight'].float().numpy().T + head['bias'].float().numpy())
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
'''9. Training feature cache
Optional reuse checks official train bytes, row IDs, labels, pinned metadata/heads, model structure and cache checksums.
The cache does not turn full-fit backbone features into out-of-fold features.
Exports are about 105 MiB for both 2699 x 5120 float32 training matrices, plus test features.
First runs intentionally extract official-training features in both iterations; that adds work versus test-only author inference.'''

In [ ]:
class TrainingFeatureCache:
    def __init__(self, output_root, signature, train):
        self.output_root = output_root
        self.signature = signature
        self.ids = train['id'].astype(str).tolist()
        self.files = {}

    def load(self, root, key):
        root = Path(root)
        manifest = json.loads((root / 'training_cache_manifest.json').read_text())
        if manifest['signature'] != self.signature or manifest['ids'] != self.ids:
            raise ValueError('Training cache uses different data, labels, token metadata or model structure')
        path = root / manifest['files'][key]['name']
        if FileChecks.sha256(path) != manifest['files'][key]['sha256']:
            raise ValueError(f'Training feature checksum mismatch: {path}')
        features = np.load(path, allow_pickle=False)
        if features.shape != (len(self.ids), 5120) or not np.isfinite(features).all():
            raise ValueError('Unexpected training feature cache shape or non-finite values')
        return features

    def save(self, key, features):
        path = self.output_root / f'{key}_training_last_layer.npy'
        np.save(path, features, allow_pickle=False)
        self.files[key] = dict(name=path.name, sha256=FileChecks.sha256(path))
        manifest = dict(signature=self.signature, ids=self.ids, files=self.files,
                        scope='Official training features only, fixed full-fit adapters; not model OOF',
                        fingerprint_scope='Pinned metadata/heads, actual adapter SHA256 plus base config and weight shard names/sizes; base weights are not fully hashed')
        (self.output_root / 'training_cache_manifest.json').write_text(json.dumps(manifest, indent=2))

In [ ]:
'''10. Fold-local readouts
A 64-dimensional feature PCA feeds a ridge classifier.
The residual candidate subtracts a ridge prediction from 32-dimensional content TF-IDF/SVD.
The fixed softmax temperature is 0.5; residual ridge and classifier alpha are 10. No hyperparameter search.'''

In [ ]:
class FeatureReadout:
    def __init__(self, residual=False):
        self.residual = residual

    def fit(self, features, texts, targets):
        self.scaler = StandardScaler()
        scaled = self.scaler.fit_transform(features)
        self.projection = PCA(n_components=min(64, scaled.shape[1], len(scaled) - 1), svd_solver='randomized', random_state=42)
        projected = self.projection.fit_transform(scaled)
        if self.residual:
            self.vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_features=10000, sublinear_tf=True)
            content = self.vectorizer.fit_transform(list(texts))
            if content.shape[1] < 2:
                raise ValueError('Insufficient content vocabulary for residual readout')
            self.content_projection = TruncatedSVD(n_components=min(32, content.shape[1] - 1, len(texts) - 1), random_state=42)
            content = self.content_projection.fit_transform(content)
            self.content_regression = Ridge(alpha=10.0).fit(content, projected)
            projected = projected - self.content_regression.predict(content)
        self.classifier = RidgeClassifier(alpha=10.0, class_weight='balanced').fit(projected, targets)
        if not np.array_equal(self.classifier.classes_, np.arange(len(LABELS))):
            raise ValueError('Every readout training fold must contain all 18 classes')
        return self

    def predict(self, features, texts):
        projected = self.projection.transform(self.scaler.transform(features))
        if self.residual:
            content = self.content_projection.transform(self.vectorizer.transform(list(texts)))
            projected = projected - self.content_regression.predict(content)
        probabilities = ProbabilityMath.softmax(np.asarray(self.classifier.decision_function(projected), dtype=np.float64) / 0.5)
        ProbabilityMath.check(probabilities, len(features))
        return probabilities

In [ ]:
'''11. Predeclared proxy validation gate
Compare plain readout against 90% plain + 10% residual on three prompt-grouped folds.
Require no negative accuracy fold, two positive folds, mean gain >= 0.005, and no mean log-loss regression.
This tests a readout proxy; it does not estimate a 10% blend gain against the complete published baseline.
On rejection the selected file remains the baseline. Inspect changed_labels_vs_baseline before spending a submission.'''

In [ ]:
class ReadoutResearch:
    def __init__(self, config):
        self.config = config

    @staticmethod
    def accept(folds):
        return (len(folds) == 3 and all(row['accuracy_delta'] >= 0 for row in folds)
                and sum(row['accuracy_delta'] > 0 for row in folds) >= 2
                and np.mean([row['accuracy_delta'] for row in folds]) >= 0.005
                and np.mean([row['log_loss_delta'] for row in folds]) <= 0)

    def fit_predict(self, train_features, test_features, train, test):
        targets = train['model'].map({label: index for index, label in enumerate(LABELS)}).to_numpy()
        splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=self.config.seed)
        plain_oof = np.zeros((len(train), len(LABELS)))
        residual_oof = np.zeros_like(plain_oof)
        fold_ids = np.full(len(train), -1, dtype=int)
        folds = []
        for fold, (fit_rows, validation_rows) in enumerate(splitter.split(train_features, targets, train['prompt_id'])):
            if set(train.iloc[fit_rows]['prompt_id']) & set(train.iloc[validation_rows]['prompt_id']):
                raise ValueError('Prompt leakage across the readout split')
            if len(np.unique(targets[fit_rows])) != len(LABELS):
                raise ValueError('A readout fit fold is missing author classes')
            fits = train.iloc[fit_rows]['text'].to_numpy()
            validation = train.iloc[validation_rows]['text'].to_numpy()
            for residual, destination in [(False, plain_oof), (True, residual_oof)]:
                readout = FeatureReadout(residual).fit(train_features[fit_rows], fits, targets[fit_rows])
                destination[validation_rows] = readout.predict(train_features[validation_rows], validation)
            mixed = (1 - self.config.readout_weight) * plain_oof[validation_rows] + self.config.readout_weight * residual_oof[validation_rows]
            plain = plain_oof[validation_rows]
            record = dict(fold=fold, rows=len(validation_rows),
                plain_accuracy=accuracy_score(targets[validation_rows], plain.argmax(1)),
                mixed_accuracy=accuracy_score(targets[validation_rows], mixed.argmax(1)),
                accuracy_delta=accuracy_score(targets[validation_rows], mixed.argmax(1)) - accuracy_score(targets[validation_rows], plain.argmax(1)),
                log_loss_delta=log_loss(targets[validation_rows], mixed, labels=np.arange(len(LABELS))) - log_loss(targets[validation_rows], plain, labels=np.arange(len(LABELS))))
            folds.append(record)
            fold_ids[validation_rows] = fold
            print(f'Readout fold {fold + 1}/3: {json.dumps(record)}', flush=True)
        if (fold_ids < 0).any():
            raise ValueError('Incomplete readout OOF coverage')
        accepted = self.accept(folds)
        report = dict(accepted=accepted, folds=folds, fixed_candidate_weight=self.config.readout_weight,
            criterion='All 3 fold accuracy deltas nonnegative, 2 positive, mean gain >= 0.005 and mean log-loss delta <= 0',
            validation_scope='Plain versus plain+residual readout proxy on fixed full-fit pretrained features; NOT full-ensemble OOF',
            limitation='Published adapters were fine-tuned on official rows. This split isolates new readout fitting, not backbone training.',
            threshold_tuned=False)
        (self.config.output_root / 'readout_validation.json').write_text(json.dumps(report, indent=2))
        np.savez_compressed(self.config.output_root / 'readout_oof.npz', ids=train['id'].to_numpy(dtype=str),
            labels=np.asarray(LABELS, dtype=str), targets=targets, fold_ids=fold_ids, plain=plain_oof, residual=residual_oof)
        if not accepted:
            print('READOUT GATE REJECTED: submission remains the Part Two baseline; do not spend a duplicate submission.', flush=True)
            return None, report
        helper = FeatureReadout(True).fit(train_features, train['text'].to_numpy(), targets)
        return helper.predict(test_features, test['text'].to_numpy()), report

In [ ]:
class ReadoutGateChecks:
    @staticmethod
    def run():
        accepted = [{'accuracy_delta': 0.01, 'log_loss_delta': -0.01} for _ in range(3)]
        assert ReadoutResearch.accept(accepted)
        rejected = [dict(row) for row in accepted]
        rejected[0]['accuracy_delta'] = -0.001
        assert not ReadoutResearch.accept(rejected)
        assert not ReadoutResearch.accept([{'accuracy_delta': 0.0, 'log_loss_delta': -0.1} for _ in range(3)])

ReadoutGateChecks.run()


In [ ]:
'''12. Submission and experiment orchestration'''

In [ ]:
class SubmissionWriter:
    def __init__(self, root):
        self.root = root

    def write(self, test, probabilities, name='submission.parquet'):
        ProbabilityMath.check(probabilities, len(test))
        frame = pd.DataFrame({'id': test['id'].astype(str), 'model': np.asarray(LABELS)[probabilities.argmax(1)]})
        if frame['id'].isna().any() or frame['id'].duplicated().any() or not frame['model'].isin(LABELS).all():
            raise ValueError('Invalid submission IDs or model labels')
        path = self.root / name
        frame.to_parquet(path, index=False)
        pd.testing.assert_frame_equal(frame, pd.read_parquet(path))
        return frame

In [ ]:
class PartTwoExperiment:
    def __init__(self, config, assets, train, test, pool):
        self.config, self.assets = config, assets
        self.train, self.test, self.pool = train, test, pool
        self.extractor = LoRAFeatureExtractor()
        self.timer = StageTimer(config.output_root)

    def run(self):
        if (self.config.output_root / 'submission.parquet').exists():
            raise FileExistsError('Start a clean run; submission.parquet already exists')
        if torch.cuda.device_count() != 2 or any('T4' not in torch.cuda.get_device_name(index) for index in range(2)):
            raise RuntimeError('Select GPU T4 x2; input-only checks can be run separately on CPU')
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        started = time.monotonic()
        try:
            with threadpool_limits(limits=4):
                return self.predict(started)
        except BaseException:
            path = self.config.output_root / 'submission.parquet'
            if path.exists():
                path.rename(path.with_name('failed_submission.parquet'))
            raise

    def predict(self, started):
        train, test = self.train, self.test
        members = {}
        with self.timer.measure('SVC on official train plus pool_v4'):
            svc = StyleClassifier().fit_predict(train, test, self.pool)
            ProbabilityMath.check(svc, len(test))
            logits = np.log(np.clip(svc, 1e-9, 1))
            members['svc'] = svc
        del self.pool
        with self.timer.measure('Original average-linkage topic groups'):
            groups, eligible = TopicGrouping.detect(test['text'].tolist())
        training_parts, test_parts = [], []
        cache = TrainingFeatureCache(self.config.output_root, self.assets.feature_signature(), train)
        for key, settings in self.config.loras.items():
            cached = (cache.load(self.config.training_cache_root, key) if self.config.training_cache_root is not None else None)
            with self.timer.measure(key + ': local staging'):
                directory = LocalCheckpointStage.copy(self.assets.assets / key, self.config.scratch_root / key)
            with self.timer.measure(key + ': test inference and optional training feature export'):
                features, training, heads = self.extractor.extract(directory, self.assets.bases[key], test['text'].tolist(),
                    train['text'].tolist() if cached is None else None, settings['bs'])
                if cached is not None:
                    training = cached
                plain = ProbabilityMath.softmax(features @ heads['plain.weight'].T + heads['plain.bias'])
                ProbabilityMath.check(plain, len(test))
                logits += settings['w'] * np.log(np.clip(plain, 1e-9, 1))
                members[key] = plain
                if eligible.any():
                    centered = features.copy()
                    for group in np.unique(groups[eligible]):
                        centered[groups == group] -= centered[groups == group].mean(0)
                    twin = ProbabilityMath.softmax(centered[eligible] @ heads['centered.weight'].T + heads['centered.bias'])
                    logits[eligible] += settings['twin_w'] * np.log(np.clip(twin, 1e-9, 1))
                    full_twin = np.full_like(plain, np.nan)
                    full_twin[eligible] = twin
                    members[key + '_twin'] = full_twin
                    del centered
                cache.save(key, training)
                training_parts.append(training)
                test_parts.append(features[:, -settings['base']['hidden_size']:].copy())
                del features, heads, cached
        with self.timer.measure('Fine-tuned DeBERTa local staging and inference'):
            directory = LocalCheckpointStage.copy(self.assets.assets / 'deberta_v3_large', self.config.scratch_root / 'deberta_v3_large')
            probabilities = DebertaClassifier().predict(directory, test['text'].tolist())
            ProbabilityMath.check(probabilities, len(test))
            logits += 0.25 * np.log(np.clip(probabilities, 1e-9, 1))
            members['deberta'] = probabilities
        baseline_raw = ProbabilityMath.softmax(logits)
        selected_raw = baseline_raw
        validation = None
        if self.config.research:
            with self.timer.measure('CPU prompt-grouped readout validation'):
                train_features = np.concatenate(training_parts, axis=1)
                test_features = np.concatenate(test_parts, axis=1)
                helper, validation = ReadoutResearch(self.config).fit_predict(train_features, test_features, train, test)
                if helper is not None:
                    selected_raw = (1 - self.config.readout_weight) * baseline_raw + self.config.readout_weight * helper
                    members['residual_helper'] = helper
        with self.timer.measure('One final Sinkhorn and submission validation'):
            writer = SubmissionWriter(self.config.output_root)
            if self.config.research:
                writer.write(test, ProbabilityMath.balance(baseline_raw), 'submission_baseline.parquet')
            selected = ProbabilityMath.balance(selected_raw)
            submission = writer.write(test, selected)
            np.savez_compressed(self.config.output_root / 'part_two_members.npz', ids=test['id'].to_numpy(dtype=str),
                labels=np.asarray(LABELS, dtype=str), groups=groups, eligible=eligible,
                baseline_raw=baseline_raw, selected_raw=selected_raw, probabilities=selected, **members)
            train[['id', 'prompt_id', 'model', 'text']].to_parquet(self.config.output_root / 'training_feature_rows.parquet', index=False)
            for key, values in zip(self.config.loras, test_parts):
                np.save(self.config.output_root / f'{key}_test_last_layer.npy', values, allow_pickle=False)
            report = dict(mode='part_two_residual_readout' if self.config.research else 'part_two_baseline',
                source_url=SOURCE_URL, source_sha256=SOURCE_SHA256, rows=len(test), training_rows=len(train),
                training_eos=self.extractor.token_audit, input_csv_sha256=self.assets.data_hashes,
                source_reported_score=0.620, score_reproduced=False, research_gate=validation,
                changed_labels_vs_baseline=int((selected.argmax(1) != ProbabilityMath.balance(baseline_raw).argmax(1)).sum()),
                elapsed_seconds=time.monotonic() - started,
                submission_sha256=FileChecks.sha256(self.config.output_root / 'submission.parquet'),
                training_features_are_backbone_oof=False,
                no_hidden_test_cache_reuse=True)
            (self.config.output_root / 'experiment_audit.json').write_text(json.dumps(report, indent=2))
            print(json.dumps({key: report[key] for key in ['mode', 'rows', 'changed_labels_vs_baseline', 'elapsed_seconds']}, indent=2))
        return submission

In [ ]:
'''13. Execute
A fresh GPU T4 x2 session is required. The model set contains no Qwen3.5, ATLAS or old complementary stack.
Save Version / Run All, then select a successfully completed version that outputs submission.parquet.
CPU-only preflight produces no submission. Do not submit an input-check-only version.
The runtime and score are unmeasured. Extra training feature extraction and CPU validation are not free.
If scratch or RAM is insufficient, the notebook fails rather than silently skipping a required member.'''

In [ ]:
if config.run_inference:
    experiment = PartTwoExperiment(config, assets, train, test, pool)
    submission = experiment.run()
    print(submission.head())
else:
    print('CPU INPUT CHECKS COMPLETE: no model inference and no submission file produced.')

## Artifacts and validation status

`input_manifest.json` records resolved checkpoints and EOS. `stage_timings.json` records running/completed/failed stages. `part_two_members.npz` stores current-pool predictions and IDs. `training_cache_manifest.json` plus both `*_training_last_layer.npy` files support optional training-cache reuse. `training_feature_rows.parquet` records the corresponding official rows.

The research version also writes `readout_validation.json`, `readout_oof.npz` and `submission_baseline.parquet`. Gate rejection or zero changed labels means there is no distinct candidate to submit for that pool.

Authored and CPU-tested locally. Full transformer/CUDA inference, notebook rendering on Kaggle, hidden runtime and leaderboard scores remain unverified. No automatic Kaggle launch is performed.

## Credits

Public passages, CC BY 4.0 unless another license is given.

- [seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages)
- [willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest)
- [lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1)
- [prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
- [alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations)
- [haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages)
- [kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages)
- [lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations)
- [abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations)
- [dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations)
- [ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations)
- [keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1)
- [michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1)
- [jadkutbi/ntl-own-proxy-generations](https://www.kaggle.com/datasets/jadkutbi/ntl-own-proxy-generations)
- [pardheev/ntl-proxy-extra-passages](https://www.kaggle.com/datasets/pardheev/ntl-proxy-extra-passages)
- [abhinavsaipodugu/ntl-proxy-gpt-gemini-passages](https://www.kaggle.com/datasets/abhinavsaipodugu/ntl-proxy-gpt-gemini-passages)
- [zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations](https://www.kaggle.com/datasets/zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations)
- [zbigniewkirchner/ntl-extra-gpt-passages](https://www.kaggle.com/datasets/zbigniewkirchner/ntl-extra-gpt-passages)
- [arxitekton/ntl-claude-gen-v1](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v1)
- [arxitekton/ntl-claude-gen-v2](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v2)
- [maniceet/name-that-llm-openrouter-openai-generations](https://www.kaggle.com/datasets/maniceet/name-that-llm-openrouter-openai-generations)
- [dedruck/ntl-proxy-gen](https://www.kaggle.com/datasets/dedruck/ntl-proxy-gen)
- [prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1), CC BY-SA 4.0
- [drjcfmd/ntl-generated-v2](https://www.kaggle.com/datasets/drjcfmd/ntl-generated-v2), CC BY-SA 4.0
- [illiadolenko/ntl-generated-corpus-v1](https://www.kaggle.com/datasets/illiadolenko/ntl-generated-corpus-v1), CC0
- [aswathvanan/ntl-proxy-passages](https://www.kaggle.com/datasets/aswathvanan/ntl-proxy-passages), CC BY-SA 4.0, only in stage one of the Qwen3-14B LoRA

Ideas and recipes.

- The LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7)
- The style aware n-gram views from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
- Adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering)

Models. Qwen3-14B-Base and Qwen2.5-14B by the Qwen team, Apache 2.0. DeBERTa-v3-large by Microsoft, MIT.